# Silver — Customers (SCD1)
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{catalog}.bronze.customers` |
| **Target** | `{catalog}.silver.customers` |
| **SCD Type** | SCD1 — latest profile always overwrites previous |
| **Depends on** | Bronze Customers task must complete first |

**Same notebook, both loads:**
Batch 1 — inserts all valid customers.
Batch 2 — updates changed profiles (email, city), inserts new customers. No code change needed.

**DQ checks applied:**
- Deduplicate rows with the same `customer_id` — keep latest by `_ingested_at`
- Exclude rows with malformed email (missing `@`)
- Replace NULL or empty `state` with `Unknown`

## Setup — Widgets & Constants

Widget values are overridden at runtime by Workflow job parameters.

In [ ]:
dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'bronze')
dbutils.widgets.text('target_schema', 'silver')

CATALOG       = dbutils.widgets.get('catalog')
SOURCE_SCHEMA = dbutils.widgets.get('source_schema')
TARGET_SCHEMA = dbutils.widgets.get('target_schema')
SOURCE_TABLE  = f'{CATALOG}.{SOURCE_SCHEMA}.customers'
TABLE         = f'{CATALOG}.{TARGET_SCHEMA}.customers'

print(f'Source : {SOURCE_TABLE}')
print(f'Target : {TABLE}')

## Step 1 — Read & Deduplicate from Bronze

Bronze accumulates all batches via COPY INTO — on the second run it holds rows from both loads.
Deduplicate by `customer_id`, keeping the latest row by `_ingested_at` so the MERGE always sees the freshest profile.

In [ ]:
from pyspark.sql.functions import col, row_number, desc
from pyspark.sql.window import Window

bronze_df = spark.table(SOURCE_TABLE)

w = Window.partitionBy('customer_id').orderBy(desc('_ingested_at'))
deduped = bronze_df.withColumn('_rn', row_number().over(w)) \
                   .filter(col('_rn') == 1) \
                   .drop('_rn', '_ingested_at', '_source_file', '_batch_id')

print(f'Bronze rows  : {bronze_df.count()}')
print(f'After dedup  : {deduped.count()}')

## Step 2 — DQ Checks

In [ ]:
from pyspark.sql.functions import when, lit

# Email must contain '@'
valid   = deduped.filter(col('email').contains('@'))
invalid = deduped.filter(~col('email').contains('@'))

print(f'Valid email   : {valid.count()}')
print(f'Invalid email : {invalid.count()} — excluded from Silver')
if invalid.count() > 0:
    invalid.select('customer_id', 'email').display()

# NULL or empty state → 'Unknown'
processed = valid.withColumn('state',
    when(col('state').isNull() | (col('state') == ''), lit('Unknown'))
    .otherwise(col('state')))

## Step 3 — Prepare Silver Columns

In [ ]:
from pyspark.sql.functions import concat_ws, to_date

silver_df = processed \
    .withColumn('full_name',  concat_ws(' ', col('first_name'), col('last_name'))) \
    .withColumn('created_at', to_date(col('created_at'))) \
    .select('customer_id', 'full_name', 'first_name', 'last_name',
            'email', 'city', 'state', 'created_at')

print(f'Rows ready for Silver: {silver_df.count()}')
silver_df.display()

## Step 4 — Create Silver Table (first run only)

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}')

spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {TABLE} (
        customer_id STRING,
        full_name   STRING,
        first_name  STRING,
        last_name   STRING,
        email       STRING,
        city        STRING,
        state       STRING,
        created_at  DATE
    )
    USING DELTA
''')

print(f'Table ready: {TABLE}')

## Step 5 — SCD1 MERGE

Update if `customer_id` already exists (profile may have changed), insert if new.
Works identically on batch 1 and batch 2 — no code change needed between runs.

In [ ]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, TABLE)

target.alias('t').merge(
    silver_df.alias('s'),
    't.customer_id = s.customer_id'
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print('MERGE complete')

## Step 6 — Verify

In [ ]:
result = spark.table(TABLE)
print(f'Total rows in {TABLE}: {result.count()}')
result.display()